In [0]:
# importing the class that is used to create and manage the spark session
from pyspark.sql import SparkSession
#importing built-in columns for aggregation
from pyspark.sql.functions import col, count, sum, avg, min, max, when
from pyspark.sql.types import TimestampType, IntegerType, DoubleType
# creating the spark session
spark = SparkSession.builder.appName("Celebal_Assignment_5").getOrCreate()

In [0]:
#Loading table sales into spark dataframe
df=spark.table("sales")
#Showing the first 5 rows of the dataframe
df.show(5)
#Displaying the column names present in the dataframe
print(df.columns)
#Printing the schema of the dataframe
df.printSchema()
#Displaying the number of rows present in the dataframe
print("Total number of rows in the dataset=", df.count())

+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|transaction_id|user_id|      username|               email|age|subscription|   transaction_date|      raw_timestamp|region|     city|product_category|store_id|               price|quantity|        sale_amount|  status|
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|        T00742|  U1194| rohansingh194|rohansingh194@exa...| 30|       Basic|2022-08-30 00:00:00|2022-08-30 00:00:00| South|Bangalore|         Grocery|   ST018| 318.649999999999980|       4|1274.59999999999990| pending|
|        T00755|  U1066|  kavyareddy66|kavyareddy66@exam...| 32|       Basic|2023-04-22 00:00:00|2023-04-22 00:00:00| So

In [0]:
#Removing duplicates in combination of 2 columns : user_id and transaction_date
duplicate_removal=df.dropDuplicates(["user_id", "transaction_date"])
#Number of rows present before removing the duplicates
print("Before:", df.count())
#Number of rows present after removing the duplicates
print("After:", duplicate_removal.count())
#Displaying first 5 rows of the newly formed dataframe with duplicates being removed
duplicate_removal.show(5)

Before: 875
After: 850
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|transaction_id|user_id|      username|               email|age|subscription|   transaction_date|      raw_timestamp|region|     city|product_category|store_id|               price|quantity|        sale_amount|  status|
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|        T00742|  U1194| rohansingh194|rohansingh194@exa...| 30|       Basic|2022-08-30 00:00:00|2022-08-30 00:00:00| South|Bangalore|         Grocery|   ST018| 318.649999999999980|       4|1274.59999999999990| pending|
|        T00755|  U1066|  kavyareddy66|kavyareddy66@exam...| 32|       Basic|2023-04-22 00:00:00|

In [0]:
#Creating a new dataframe df_sales which is a copy of duplicate_removal dataframe
df_sales=duplicate_removal
#Writing the query to find average sale_amount per product_category where region=West
df_sales=df_sales.filter(col("region")=="West").groupby("product_category").agg(avg("sale_amount").alias("avg_sale_amount"))
df_sales.show()

+----------------+--------------------+
|product_category|     avg_sale_amount|
+----------------+--------------------+
|     Electronics|2638.555000000000...|
|         Grocery|2663.019487179487...|
|          Sports|3047.578541666666...|
|       Furniture|3542.441621621621...|
|        Clothing|3063.731176470588...|
+----------------+--------------------+



In [0]:
#Using .na.fill()
df_fill=duplicate_removal.na.fill({"status": "Unknown"})
df_fill.show(5)


+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|transaction_id|user_id|      username|               email|age|subscription|   transaction_date|      raw_timestamp|region|     city|product_category|store_id|               price|quantity|        sale_amount|  status|
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|        T00742|  U1194| rohansingh194|rohansingh194@exa...| 30|       Basic|2022-08-30 00:00:00|2022-08-30 00:00:00| South|Bangalore|         Grocery|   ST018| 318.649999999999980|       4|1274.59999999999990| pending|
|        T00755|  U1066|  kavyareddy66|kavyareddy66@exam...| 32|       Basic|2023-04-22 00:00:00|2023-04-22 00:00:00| So

In [0]:
#Using .na.drop()
df_drop=duplicate_removal.na.drop(subset=["status"])
print("number of rows left after dropping nulls in status:", df_drop.count())

number of rows left after dropping nulls in status: 633


In [0]:
#Checking if there are any negative values in the columns price, sale_amount and age, as if negative values are present then that means that the data is incorrect as price can never be less the 0 in the same way sale amount and age also cannot be less than 0
duplicate_removal.filter((col("price") < 0)|(col("sale_amount")<0)|(col("age") < 0)).show()
#As we can see that the columns are empty, hence there is no incorrect/ inconsistenty data present in the dataframe.

+--------------+-------+--------+-----+---+------------+----------------+-------------+------+----+----------------+--------+-----+--------+-----------+------+
|transaction_id|user_id|username|email|age|subscription|transaction_date|raw_timestamp|region|city|product_category|store_id|price|quantity|sale_amount|status|
+--------------+-------+--------+-----+---+------------+----------------+-------------+------+----+----------------+--------+-----+--------+-----------+------+
+--------------+-------+--------+-----+---+------------+----------------+-------------+------+----+----------------+--------+-----+--------+-----------+------+



In [0]:
#Grouping by city and only displaying names of those cities whose total number of records is greater than 100
df6= duplicate_removal.groupBy("city").count().filter(col("count")>100)
df6.show()
#As there is no city like this lets take it greater than 50
df6= duplicate_removal.groupBy("city").count().filter(col("count")>50)
df6.show()

+----+-----+
|city|count|
+----+-----+
+----+-----+

+----------+-----+
|      city|count|
+----------+-----+
| Bangalore|   78|
|   Chennai|   65|
|     Delhi|   73|
|   Lucknow|   66|
|Chandigarh|   71|
| Ahmedabad|   79|
| Hyderabad|   79|
|   Kolkata|   67|
|  Guwahati|   70|
|      Pune|   65|
|     Patna|   77|
|    Mumbai|   60|
+----------+-----+



In [0]:
df8=duplicate_removal.filter((col("age")>=18) & (col("age")<=30) & (col("subscription")=="Premium"))
df8.show(5)

+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|transaction_id|user_id|      username|               email|age|subscription|   transaction_date|      raw_timestamp|region|     city|product_category|store_id|               price|quantity|        sale_amount|  status|
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|        T00003|  U1071|  karangupta71|karangupta71@exam...| 18|     Premium|2024-01-18 00:00:00|2024-01-18 00:00:00| North|  Lucknow|     Electronics|   ST010|1587.790000000000000|       1|1587.79000000000000|inactive|
|        T00227|  U1244|rohansharma244|rohansharma244@ex...| 20|     Premium|2023-05-15 00:00:00|2023-05-15 00:00:00| No

In [0]:
duplicate_removal.select(avg("sale_amount")).show()
print("Total rows:", duplicate_removal.count())
print("Non-null sale_amount rows:", duplicate_removal.filter(col("sale_amount").isNotNull()).count())

+--------------------+
|    avg(sale_amount)|
+--------------------+
|3097.374024539877...|
+--------------------+

Total rows: 850
Non-null sale_amount rows: 815


In [0]:
from pyspark.sql.types import TimestampType
df10= duplicate_removal.withColumn("raw_timestamp", col("raw_timestamp").cast(TimestampType())).withColumnRenamed("raw_timestamp", "event_time")
df10.printSchema()
df10.select("event_time").show(10)
df10.filter(col("event_time").isNull()).count()

root
 |-- transaction_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: long (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- region: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- price: decimal(35,15) (nullable = true)
 |-- quantity: long (nullable = true)
 |-- sale_amount: decimal(34,14) (nullable = true)
 |-- status: string (nullable = true)

+-------------------+
|         event_time|
+-------------------+
|2022-08-30 00:00:00|
|2023-04-22 00:00:00|
|2024-02-18 00:00:00|
|2024-01-18 00:00:00|
|2023-10-27 00:00:00|
|2022-06-06 00:00:00|
|2023-05-15 00:00:00|
|2023-10-23 00:00:00|
|2023-10-11 00:00:00|
|2022-08-14 00:00:00|
+-------------------+
only showing top 

0

In [0]:
df12 = duplicate_removal.filter(col("email").isNotNull() & (col("email")!="") & (col("username")!=""))
print("Before:", duplicate_removal.count())
print("After:", df12.count())
df12.show(5)

Before: 850
After: 814
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|transaction_id|user_id|      username|               email|age|subscription|   transaction_date|      raw_timestamp|region|     city|product_category|store_id|               price|quantity|        sale_amount|  status|
+--------------+-------+--------------+--------------------+---+------------+-------------------+-------------------+------+---------+----------------+--------+--------------------+--------+-------------------+--------+
|        T00742|  U1194| rohansingh194|rohansingh194@exa...| 30|       Basic|2022-08-30 00:00:00|2022-08-30 00:00:00| South|Bangalore|         Grocery|   ST018| 318.649999999999980|       4|1274.59999999999990| pending|
|        T00755|  U1066|  kavyareddy66|kavyareddy66@exam...| 32|       Basic|2023-04-22 00:00:00|

In [0]:
df13 = duplicate_removal.agg(min("price").alias("min_price"),max("price").alias("max_price"),avg("price").alias("mean_price"))
df13.show()

+------------------+--------------------+--------------------+
|         min_price|           max_price|          mean_price|
+------------------+--------------------+--------------------+
|53.810000000000000|1995.680000000000000|1029.952219541616...|
+------------------+--------------------+--------------------+



In [0]:
df_pipeline = duplicate_removal.na.fill({"price": 0})
df_pipeline = df_pipeline.groupBy("store_id").agg(sum("sale_amount").alias("total_revenue"))
df_pipeline.show(5)

+--------+--------------------+
|store_id|       total_revenue|
+--------+--------------------+
|   ST018|78887.91000000000020|
|   ST005|96423.64999999999964|
|   ST001|104327.0899999999...|
|   ST010|86497.36000000000015|
|   ST015|92362.05000000000040|
+--------+--------------------+
only showing top 5 rows
